In [ ]:
import urllib.request
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D,
    Conv2DTranspose, concatenate
)

print("Libraries imported successfully!")

In [ ]:
url = 'https://images.unsplash.com/photo-1557050543-4d5f4e07ef46?auto=format&fit=crop&w=500&q=80'

headers = {'User-Agent': 'Mozilla/5.0'}

try:
    req = urllib.request.Request(url, headers=headers)

    with urllib.request.urlopen(req) as response:
        img_array = np.asarray(
            bytearray(response.read()), dtype=np.uint8
        )

    real_img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    if real_img is None:
        raise ValueError("Could not decode image.")

    gray = cv2.cvtColor(real_img, cv2.COLOR_BGR2GRAY)

    _, mask_src = cv2.threshold(
        gray, 110, 255, cv2.THRESH_BINARY_INV
    )

    img_resized = cv2.resize(gray, (128, 128))
    mask_resized = cv2.resize(mask_src, (128, 128))

    img_norm = np.expand_dims(
        img_resized.astype(np.float32) / 255.0, axis=-1
    )

    mask_norm = np.expand_dims(
        (mask_resized > 127).astype(np.float32), axis=-1
    )

    images = np.array([img_norm] * 100, dtype=np.float32)
    masks = np.array([mask_norm] * 100, dtype=np.float32)

    def augment(image, mask):
        if tf.random.uniform(()) > 0.5:
            image = tf.image.flip_left_right(image)
            mask = tf.image.flip_left_right(mask)

        image = tf.image.random_brightness(image, max_delta=0.15)
        image = tf.clip_by_value(image, 0.0, 1.0)

        return image, mask

    elephant_ds = tf.data.Dataset.from_tensor_slices(
        (images, masks)
    )

    elephant_ds = elephant_ds.map(augment).shuffle(100)

    train_gen = elephant_ds.take(80).batch(4).prefetch(
        tf.data.AUTOTUNE
    )

    test_gen = elephant_ds.skip(80).batch(4).prefetch(
        tf.data.AUTOTUNE
    )

    print("Dataset prepared successfully!")

except Exception as e:
    print(f"Error fetching image: {e}")

In [ ]:
def full_dice_coef(y_true, y_pred, smooth=1e-6):

    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)

    intersection = tf.reduce_sum(y_true_f * y_pred_f)

    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true_f) +
        tf.reduce_sum(y_pred_f) + smooth
    )


def full_jaccard_index(y_true, y_pred, smooth=1e-6):

    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)

    intersection = tf.reduce_sum(y_true_f * y_pred_f)

    union = (
        tf.reduce_sum(y_true_f) +
        tf.reduce_sum(y_pred_f) - intersection
    )

    return (intersection + smooth) / (union + smooth)

In [ ]:
def build_full_unet(input_shape=(128, 128, 1)):

    inputs = Input(input_shape)

    c1 = Conv2D(64, 3, activation='relu', padding='same')(inputs)
    c1 = Conv2D(64, 3, activation='relu', padding='same')(c1)
    p1 = MaxPooling2D(2)(c1)

    c2 = Conv2D(128, 3, activation='relu', padding='same')(p1)
    c2 = Conv2D(128, 3, activation='relu', padding='same')(c2)
    p2 = MaxPooling2D(2)(c2)

    c3 = Conv2D(256, 3, activation='relu', padding='same')(p2)
    c3 = Conv2D(256, 3, activation='relu', padding='same')(c3)
    p3 = MaxPooling2D(2)(c3)

    c4 = Conv2D(512, 3, activation='relu', padding='same')(p3)
    c4 = Conv2D(512, 3, activation='relu', padding='same')(c4)
    p4 = MaxPooling2D(2)(c4)

    c5 = Conv2D(1024, 3, activation='relu', padding='same')(p4)
    c5 = Conv2D(1024, 3, activation='relu', padding='same')(c5)

    u6 = Conv2DTranspose(512, 2, strides=2, padding='same')(c5)
    u6 = concatenate([u6, c4])
    c6 = Conv2D(512, 3, activation='relu', padding='same')(u6)
    c6 = Conv2D(512, 3, activation='relu', padding='same')(c6)

    u7 = Conv2DTranspose(256, 2, strides=2, padding='same')(c6)
    u7 = concatenate([u7, c3])
    c7 = Conv2D(256, 3, activation='relu', padding='same')(u7)
    c7 = Conv2D(256, 3, activation='relu', padding='same')(c7)

    u8 = Conv2DTranspose(128, 2, strides=2, padding='same')(c7)
    u8 = concatenate([u8, c2])
    c8 = Conv2D(128, 3, activation='relu', padding='same')(u8)
    c8 = Conv2D(128, 3, activation='relu', padding='same')(c8)

    u9 = Conv2DTranspose(64, 2, strides=2, padding='same')(c8)
    u9 = concatenate([u9, c1])
    c9 = Conv2D(64, 3, activation='relu', padding='same')(u9)
    c9 = Conv2D(64, 3, activation='relu', padding='same')(c9)

    outputs = Conv2D(1, 1, activation='sigmoid')(c9)

    return Model(inputs, outputs)

In [ ]:
full_model = build_full_unet()

full_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=[full_dice_coef, full_jaccard_index]
)

print("Model compiled successfully!")

In [ ]:
print("Training models...")

full_model.fit(
    train_gen,
    steps_per_epoch=20,
    validation_data=test_gen,
    validation_steps=5,
    epochs=3
)

In [ ]:
test_imgs, test_masks = next(iter(test_gen))

predictions = full_model.predict(test_imgs)

predictions_bin = (
    predictions > 0.5
).astype("float32")

print(
    "Test Dice Coefficient:",
    full_dice_coef(test_masks, predictions_bin).numpy()
)

print(
    "Test Jaccard Index:",
    full_jaccard_index(test_masks, predictions_bin).numpy()
)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.title("Real Test Image")
plt.imshow(test_imgs[0].numpy().squeeze(), cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("True Mask")
plt.imshow(test_masks[0].numpy().squeeze(), cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title("Predicted Mask")
plt.imshow(predictions_bin[0].squeeze(), cmap='gray')
plt.axis('off')

plt.show()